## Successful Semantic Modelling for Power BI - "Lab 02. Storage modes"

Module 2 in the browser. Replaces what would otherwise be a DAX Studio demo, so
nobody needs a desktop tool, the right version, or admin rights to follow along.

WHAT IT SHOWS - run the cells IN ORDER, each one builds on the last

ACT 1  Direct Lake paging (cells 4-9)
cold -> query -> more resident -> query -> more resident -> reframe -> cold
plus the trap that ClearCache does NOT undo any of it.
ACT 2  DirectQuery (cell 10) - what a DirectQueryEnd event looks like.

Hybrid (import + DirectQuery partitions) belongs to Module 4, not here. The
queries and the data-coverage trap live in handouts/dax-snippets.md.

Run as a **Python** notebook. It is all DAX, DMV and REST, no Spark session.

In [ ]:
# ---- CELL 1: INSTALL --------------------------------------------------------
# Alone, and first. In Fabric %pip restarts the Python interpreter, so anything
# defined before it is lost. Config and imports therefore live in Cell 2.
%pip install -q semantic-link-labs

In [ ]:
# ---- CELL 2: CONFIG + IMPORTS ----------------------------------------------
WORKSPACE = None                              # None = this notebook's workspace
DL_MODEL  = "01 Star Schema (fixed)"          # Direct Lake, for Act 1
DQ_MODEL  = "02 Storage - DirectQuery"        # 100% DirectQuery, for Act 2

import sempy.fabric as fabric
import sempy_labs as labs
import pandas as pd
import time
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

WORKSPACE = WORKSPACE or fabric.resolve_workspace_name()
print(f"workspace: {WORKSPACE}")

# Act 1 runs anywhere. Act 2 needs a presenter-built model that an attendee
# workspace does not have, so check once here rather than throwing mid-demo.
_present = set(fabric.list_items(workspace=WORKSPACE)
                     .query("Type=='SemanticModel'")["Display Name"])

def have(model):
    if model in _present:
        return True
    print(f"SKIPPED: '{model}' is a presenter model, not in this workspace.")
    print("         Act 1 runs fine here. Watch the screen for this one.")
    return False

for m in (DL_MODEL, DQ_MODEL):
    print(f"  {'ok ' if m in _present else 'n/a'}  {m}")

In [ ]:
# ---- CELL 3: HELPER - which columns are in memory right now? ---------------
# Direct Lake pages column dictionaries in on demand. This DMV is the ground
# truth, and unlike the segment DMV it gives real table and column names.
#   ISPAGEABLE   can this column come and go (ie is it Direct Lake backed)
#   ISRESIDENT   is it in memory right now
#   TEMPERATURE  how recently and how often it has been touched
COLUMNS_DMV = """
SELECT
    MEASURE_GROUP_NAME       AS [TABLE],
    ATTRIBUTE_NAME           AS [COLUMN],
    DATATYPE,
    DICTIONARY_SIZE          AS SIZE,
    DICTIONARY_ISPAGEABLE    AS PAGEABLE,
    DICTIONARY_ISRESIDENT    AS RESIDENT,
    DICTIONARY_TEMPERATURE   AS TEMPERATURE,
    DICTIONARY_LAST_ACCESSED AS LASTACCESSED
FROM $SYSTEM.DISCOVER_STORAGE_TABLE_COLUMNS
ORDER BY
    [DICTIONARY_TEMPERATURE] DESC
"""

def resident(model, label="", show=15):
    """Print the columns currently paged into memory. Returns the resident rows."""
    try:
        df = fabric.evaluate_dax(dataset=model, workspace=WORKSPACE, dax_string=COLUMNS_DMV)
    except Exception as e:
        print(f"  DMV unavailable ({str(e)[:120]}) - fall back to the Delta Analyzer demo")
        return pd.DataFrame()

    df.columns = [c.split("[")[-1].rstrip("]") for c in df.columns]
    df = df[df["COLUMN"].astype(str) != "RowNumber"]

    def flag(col):
        return df[col].astype(str).str.lower().isin(["true", "1"])

    pageable, hot = df[flag("PAGEABLE")], df[flag("RESIDENT")]
    print(f"{label}: {len(hot)} of {len(pageable)} pageable columns resident")
    if len(hot):
        display(hot[["TABLE", "COLUMN", "DATATYPE", "SIZE",
                     "TEMPERATURE", "LASTACCESSED"]].head(show))
    return hot


# Server Timings in a notebook. QueryEnd = the whole query, VertiPaqSEQueryEnd =
# storage engine scans, DirectQueryEnd = a round trip to the source. That last
# one is what Acts 2 and 3 are about: we want to SEE it, then make it disappear.
EVENTS = {
    "QueryEnd":           ["EventClass", "EventSubclass", "TextData", "Duration", "CpuTime"],
    "VertiPaqSEQueryEnd": ["EventClass", "EventSubclass", "TextData", "Duration", "CpuTime"],
    "DirectQueryEnd":     ["EventClass", "TextData", "Duration", "CpuTime"],
}

def _col(df, name):
    key = name.replace(" ", "").lower()
    for c in df.columns:
        if c.replace(" ", "").lower() == key:
            return c
    return None

def run(model, dax, label="query", settle=5, show_sql=False):
    """Run a DAX query with a trace and report SE scans vs DirectQuery events."""
    with fabric.create_trace_connection(dataset=model, workspace=WORKSPACE) as tc:
        with tc.create_trace(EVENTS, "Storage modes") as tr:
            tr.start()
            result = fabric.evaluate_dax(dataset=model, workspace=WORKSPACE, dax_string=dax)
            time.sleep(settle)
            logs = tr.stop()

    ec, sub, dur = _col(logs, "EventClass"), _col(logs, "EventSubclass"), _col(logs, "Duration")
    if ec is None:
        display(logs)
        return result

    q  = logs[logs[ec] == "QueryEnd"]
    se = logs[logs[ec] == "VertiPaqSEQueryEnd"]
    dq = logs[logs[ec] == "DirectQueryEnd"]
    if sub is not None:                       # drop the internal duplicates
        se = se[~se[sub].astype(str).str.contains("Internal", case=False, na=False)]

    total  = float(q[dur].max() or 0)
    dq_ms  = float(dq[dur].sum() or 0)
    verdict = "DirectQuery partition WAS queried" if len(dq) else "no DirectQuery, served from memory"
    print(f"{label}")
    print(f"  total {total:>6.0f} ms | SE scans {len(se):>2} | DirectQuery events {len(dq):>2} ({dq_ms:.0f} ms)  -> {verdict}")
    if show_sql and len(dq):
        print(dq[_col(logs, "TextData")].iloc[0][:600])
    return result

In [ ]:
# ---- CELL 4: ACT 1 step 1 - start from genuinely cold ----------------------
# Getting back to cold means REFRAMING, not clearing the cache. A reframe points
# the model at the latest Delta version and invalidates the in-memory cache, so
# every column has to be paged in again. Seconds, and no data is copied.
# Expect: 0 resident.
labs.refresh_semantic_model(dataset=DL_MODEL, workspace=WORKSPACE)
cold = resident(DL_MODEL, "1. COLD (after reframe)")

In [ ]:
# ---- CELL 5: ACT 1 step 2 - one measure, one column ------------------------
# The measure only needs sales[SalesAmount]. Nothing else should appear.
run(DL_MODEL, 'EVALUATE ROW("Total", [Total Sales])', "one measure")
warm1 = resident(DL_MODEL, "2. AFTER one measure")

In [ ]:
# ---- CELL 6: ACT 1 step 3 - add grouping columns ---------------------------
# Now we need two dimension columns and a second measure as well. PREDICT how
# many segments will be resident before you run it. That is the exercise.
run(DL_MODEL, """
EVALUATE
SUMMARIZECOLUMNS (
    'Date'[MonthYear],
    'Product'[Category],
    "Sales", [Total Sales],
    "Qty",   [Total Quantity]
)
""", "grouped query")
warm2 = resident(DL_MODEL, "3. AFTER a grouped query")

print()
print(f"resident columns: {len(cold)} -> {len(warm1)} -> {len(warm2)}")
print("Only the columns each query touched were paged in. That is column-on-demand.")

In [ ]:
# ---- CELL 7: ACT 1 step 4 - the whole model at a glance --------------------
# VertiPaq Analyzer as an interactive HTML report, right here in the notebook.
#
# read_stats_from_data=True fetches column cardinality from the data. On Direct
# Lake that means querying the Delta tables. Costs a few seconds, hence opt-in.
#
# READ THIS AS A MEMORY PICTURE. On Direct Lake it describes what VertiPaq holds
# *right now*, which is why cells 4-6 came first. For the on-DISK story (Parquet
# rowgroups, file count, V-Order) use labs.delta_analyzer, as Module 1 does.
vpa = labs.vertipaq_analyzer(dataset=DL_MODEL, workspace=WORKSPACE,
                             read_stats_from_data=True)   # dark_mode=True projects better

print("sections returned:", list(vpa))   # each one is a DataFrame you can sort and filter

In [ ]:
# ---- CELL 8: ACT 1 step 5 - ClearCache does NOT undo any of this -----------
# Two different caches, and only one of them evicts columns:
#
#   ClearCache  drops the storage engine's cached query RESULTS. Useful before
#               timing a query so you measure real work. Columns stay resident.
#   Reframe     invalidates the in-memory column cache itself.
#
# Expect the count to be UNCHANGED from cell 6.
labs.clear_cache(dataset=DL_MODEL, workspace=WORKSPACE)
after_clear = resident(DL_MODEL, "4. AFTER clear_cache")
print(f"  still {len(after_clear)} resident. Query results were dropped, columns were not.")

In [ ]:
# ---- CELL 9: ACT 1 step 6 - reframe DOES, closing the loop -----------------
# Expect 0 again, back where cell 4 started.
labs.refresh_semantic_model(dataset=DL_MODEL, workspace=WORKSPACE)
after_reframe = resident(DL_MODEL, "5. AFTER reframe")
print(f"  now {len(after_reframe)} resident. Reframing is what evicts columns.")

In [ ]:
# ---- CELL 10: ACT 2 - what DirectQuery actually looks like -----------------
# Same shape of query, but a 100% DirectQuery model. Every request becomes SQL,
# so expect DirectQuery events > 0 and no VertiPaq scans worth speaking of.
# Nothing is cached in memory, which is the trade: always current, never instant.
if have(DQ_MODEL):
    run(DQ_MODEL, """
EVALUATE
SUMMARIZECOLUMNS ( 'date'[MonthYear], "Sales", [Total Sales] )
""", "DirectQuery model", show_sql=True)

In [ ]:
# ---- CELL 11: REFERENCE - sources, fallback and guardrails -----------------
# Where the Direct Lake model reads from, whether anything would fall back, and
# the SKU limits that decide when Direct Lake stops fitting in memory.
for src in labs.directlake.get_direct_lake_sources(dataset=DL_MODEL, workspace=WORKSPACE):
    route = "SQL endpoint" if src.get("usesSqlEndpoint") else "OneLake (direct)"
    print(f"  {src.get('itemName')} ({src.get('itemType')}) -> {route}")

display(labs.directlake.check_fallback_reason(dataset=DL_MODEL, workspace=WORKSPACE))

sku = labs.directlake.get_sku_size(workspace=WORKSPACE)
print(f"Capacity SKU: {sku}")
display(labs.directlake.get_directlake_guardrails_for_sku(sku))

In [ ]:
# ---- CELL 12: A REAL GUARDRAIL BREACH --------------------------------------
# sales_guardrail_10k is 10,000 rows written one per file: ~10,000 files AND
# ~10,000 row groups against a P1 cap of 5,000 each, and 81 MB to hold what
# fits in a few hundred KB. Built deliberately - nobody writes one row per file
# by accident, and saying so is part of the demo.
#
# This is Direct Lake over ONELAKE, so there is no DirectQuery to fall back to.
# The model does not go slow. It stops answering, and the error names the
# guardrail. Measured 24 Aug 2026 on the workshop P1.
#
# Presenter model, so attendees get the skip message. Looked up live rather than
# through have(), which reads a snapshot taken back in Cell 2.
import re

BAD_MODEL = "05 File Health - partitioned 10K"
BAD_TABLE = "sales_guardrail_10k"

if BAD_MODEL not in set(fabric.list_items(workspace=WORKSPACE)
                              .query("Type=='SemanticModel'")["Display Name"]):
    print(f"SKIPPED: '{BAD_MODEL}' is a presenter model, not in this workspace.")
    print("         Watch the screen for this one.")
else:
    # TableTraits reports the breach by name. check_fallback_reason does not,
    # which is why it is not used here.
    print("1. TableTraits - the engine's own answer")
    tt = fabric.evaluate_dax(dataset=BAD_MODEL, workspace=WORKSPACE,
                             dax_string="EVALUATE TableTraits()")
    keep = [c for c in tt.columns
            if any(k in c for k in ("TableName", "StorageMode", "FallbackInfo"))]
    display(tt[keep])

    # Expected to raise. Caught so the traceback does not look like a mistake.
    print("\n2. Now actually query it")
    try:
        run(BAD_MODEL, f'EVALUATE ROW ( "Rows", COUNTROWS ( {BAD_TABLE} ) )', "10k row groups")
        print("   ...it answered. Compare the guardrails above - you may be under them.")
    except Exception as e:
        msg = str(e).replace("<oii>", "").replace("</oii>", "")
        hit = re.search(r"(We can't run a DAX query.*?learn more)", msg, re.S)
        print("   REFUSED - and this is the demo, not a mistake:\n")
        print("   " + (hit.group(1) if hit else msg[:500]))